In [ ]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.0 MB/s eta 0:00:00


In [ ]:
from torchvision import datasets

# Ładujemy metadane z pobranego już zbioru (bez transformacji, tylko po to by odczytać klasy)
temp_dataset = datasets.Food101(root='.', split='test', download=False)

# Pobieramy listę klas
potrawy = temp_dataset.classes

print(f"Łączna liczba klas: {len(potrawy)}\n")
print("Oto pełna lista klas w kolejności alfabetycznej (tak jak widzi je model):")
print("-" * 50)

# Wypisujemy klasy ładnie sformatowane (po 5 w linijce, żeby nie zajmować ekranu)
for i in range(0, len(potrawy), 5):
    print(", ".join(potrawy[i:i+5]))

Łączna liczba klas: 101

Oto pełna lista klas w kolejności alfabetycznej (tak jak widzi je model):
--------------------------------------------------
apple_pie, baby_back_ribs, baklava, beef_carpaccio, beef_tartare
beet_salad, beignets, bibimbap, bread_pudding, breakfast_burrito
bruschetta, caesar_salad, cannoli, caprese_salad, carrot_cake
ceviche, cheese_plate, cheesecake, chicken_curry, chicken_quesadilla
chicken_wings, chocolate_cake, chocolate_mousse, churros, clam_chowder
club_sandwich, crab_cakes, creme_brulee, croque_madame, cup_cakes
deviled_eggs, donuts, dumplings, edamame, eggs_benedict
escargots, falafel, filet_mignon, fish_and_chips, foie_gras
french_fries, french_onion_soup, french_toast, fried_calamari, fried_rice
frozen_yogurt, garlic_bread, gnocchi, greek_salad, grilled_cheese_sandwich
grilled_salmon, guacamole, gyoza, hamburger, hot_and_sour_soup
hot_dog, huevos_rancheros, hummus, ice_cream, lasagna
lobster_bisque, lobster_roll_sandwich, macaroni_and_cheese, macarons, 

In [ ]:
# =====================================================================
# 0. INSTALACJA BRAKUJĄCYCH BIBLIOTEK DLA ONNX
# =====================================================================
!pip install onnx onnxscript

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# Sprawdzenie dostępności GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

# =====================================================================
# 1. WYBÓR 20 NAJPOPULARNIEJSZYCH KLAS
# =====================================================================
popular_classes = [
    'apple_pie', 'beef_tartare', 'caesar_salad', 'cheesecake', 'chicken_wings',
    'chocolate_cake', 'club_sandwich', 'dumplings', 'french_fries', 'garlic_bread',
    'hamburger', 'ice_cream', 'lasagna', 'macaroni_and_cheese', 'omelette',
    'pancakes', 'pizza', 'spaghetti_bolognese', 'sushi', 'waffles'
]

# Tworzymy mapowanie: stare indeksy Food101 -> nowe indeksy (0-19)
# To jest krytyczne dla FastAPI, aby klasy szły od 0 do 19!
class_to_idx_mapping = {cls: idx for idx, cls in enumerate(popular_classes)}
print(f"Wybrane klasy ({len(popular_classes)}): {popular_classes}")

# =====================================================================
# 2. PRZYGOTOWANIE I FILTROWANIE DANYCH
# =====================================================================
print("\n--- Krok 1: Filtrowanie zbioru danych Food101 ---")

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Ładowanie pełnego zbioru (nie pobiera się na nowo, czyta z dysku Colaba)
full_train_dataset = datasets.Food101(root='.', split='train', download=False, transform=data_transforms['train'])
full_val_dataset = datasets.Food101(root='.', split='test', download=False, transform=data_transforms['val'])

# Funkcja filtrująca indeksy zdjęć należących tylko do naszych 20 klas
def filter_indices(dataset, allowed_classes, class_to_idx_map):
    indices = []
    new_labels = {}

    # Przechodzimy po wszystkich etykietach w zbiorze
    for idx, target_id in enumerate(dataset._labels):
        class_name = dataset.classes[target_id]
        if class_name in allowed_classes:
            indices.append(idx)
            # Nadpisujemy starą etykietę (0-100) nową etykietą (0-19)
            new_labels[idx] = class_to_idx_map[class_name]

    return indices, new_labels

train_indices, train_new_labels = filter_indices(full_train_dataset, popular_classes, class_to_idx_mapping)
val_indices, val_new_labels = filter_indices(full_val_dataset, popular_classes, class_to_idx_mapping)

# Tworzymy podzbiory (Subsets)
train_subset = Subset(full_train_dataset, train_indices)
val_subset = Subset(full_val_dataset, val_indices)

# Klasa pomocnicza, która podmienia etykiety w locie na te z zakresu 0-19
class ModifiedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, new_labels_dict):
        self.subset = subset
        self.new_labels_dict = new_labels_dict
    def __getitem__(self, index):
        img, _ = self.subset[index]
        actual_dataset_index = self.subset.indices[index]
        new_label = self.new_labels_dict[actual_dataset_index]
        return img, new_label
    def __len__(self):
        return len(self.subset)

train_dataset = ModifiedSubset(train_subset, train_new_labels)
val_dataset = ModifiedSubset(val_subset, val_new_labels)

# Tworzenie DataLoaderów
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
}

num_classes = len(popular_classes)
print(f"Filtrowanie zakończone. Trening: {len(train_dataset)} zdjęć, Walidacja: {len(val_dataset)} zdjęć.")

# =====================================================================
# 3. BUDOWA MODELU (TRANSFER LEARNING)
# =====================================================================
print("\n--- Krok 2: Inicjalizacja modelu MobileNetV2 ---")
try:
    model = models.mobilenet_v2(weights=models.MobileNetV2_Weights.DEFAULT)
except AttributeError:
    model = models.mobilenet_v2(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

# Nowa końcówka klasyfikatora ustawiona dokładnie na 20 klas
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.005)

# =====================================================================
# 4. SZYBKI TRENING
# =====================================================================
print("\n--- Krok 3: Rozpoczęcie szybkiego trenowania ---")
num_epochs = 3

for epoch in range(num_epochs):
    print(f'Epoka {epoch + 1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        dataset_size = len(train_dataset) if phase == 'train' else len(val_dataset)
        epoch_loss = running_loss / dataset_size
        epoch_acc = running_corrects.double() / dataset_size

        print(f'{phase.upper()} Strata: {epoch_loss:.4f} Dokładność (Accuracy): {epoch_acc:.4f}')

print('Trening na 20 klasach zakończony!')

# =====================================================================
# 5. EKSPORT DO FORMATU ONNX
# =====================================================================
print("\n--- Krok 4: Eksportowanie modelu do formatu ONNX ---")
model.eval()

dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_file_path = "../../Downloads/food_classifier_20.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}}
)

print(f"\n🚀 SUKCES! Plik '{onnx_file_path}' (20 klas) jest gotowy do pobrania!")

Używane urządzenie: cuda
Wybrane klasy (20): ['apple_pie', 'beef_tartare', 'caesar_salad', 'cheesecake', 'chicken_wings', 'chocolate_cake', 'club_sandwich', 'dumplings', 'french_fries', 'garlic_bread', 'hamburger', 'ice_cream', 'lasagna', 'macaroni_and_cheese', 'omelette', 'pancakes', 'pizza', 'spaghetti_bolognese', 'sushi', 'waffles']

--- Krok 1: Filtrowanie zbioru danych Food101 ---
Filtrowanie zakończone. Trening: 15000 zdjęć, Walidacja: 5000 zdjęć.

--- Krok 2: Inicjalizacja modelu MobileNetV2 ---

--- Krok 3: Rozpoczęcie szybkiego trenowania ---
Epoka 1/3
----------
TRAIN Strata: 1.7689 Dokładność (Accuracy): 0.5211
VAL Strata: 0.9495 Dokładność (Accuracy): 0.7248
Epoka 2/3
----------
TRAIN Strata: 1.7231 Dokładność (Accuracy): 0.5714
VAL Strata: 1.0577 Dokładność (Accuracy): 0.7192
Epoka 3/3
----------
TRAIN Strata: 1.7067 Dokładność (Accuracy): 0.5862
VAL Strata: 0.9879 Dokładność (Accuracy): 0.7348
Trening na 20 klasach zakończony!

--- Krok 4: Eksportowanie modelu do formatu 

/tmp/ipykernel_1036/1140934857.py:177: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0613 11:23:37.121000 1036 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Asserti

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

🚀 SUKCES! Plik 'food_classifier_20.onnx' (20 klas) jest gotowy do pobrania!
